# Surface Metrics and Benchmark

How do you know if a travel-time surface is accurate?  You need ground-truth
observations — GPS-tracked travel times, validated API calls, or survey data
— and a structured evaluation framework.

**What's covered**
1. Ground-truth data format and the train/test split
2. `score_at_points()` — point-based metrics
3. `compare_surfaces()` — pixel-wise inter-backend comparison
4. `run_benchmark()` — full structured benchmark pipeline
5. Residual maps — where does each backend over/under-predict?
6. Coverage error — the metric that matters most for health access
7. In-sample vs. out-of-sample performance


## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from idd_travel_time.access import (
    BoundingBox,
    EnsembleSurfaceRouter,
    FrictionSurfaceRouter,
    OsmnxSurfaceRouter,
    compare_surfaces,
    make_synthetic_ground_truth,
    residual_geodataframe,
    run_benchmark,
    score_at_points,
)
from idd_travel_time.constants import TransportMode

DESTINATION = (-1.3006, 36.8073)
BBOX = BoundingBox(west=36.76, south=-1.36, east=36.86, north=-1.24)
RESOLUTION = 0.01
MODE = TransportMode.WALKING

osmnx_router = OsmnxSurfaceRouter()
friction_5kph = FrictionSurfaceRouter(speed_kph_override=5.0)
friction_4kph = FrictionSurfaceRouter(speed_kph_override=4.0)

## 1 · Ground truth data format

In [ ]:
# In practice, ground truth comes from GPS-tracked surveys or validated APIs.
# Here we generate synthetic GT from the osmnx surface + Gaussian noise.
# This lets us demonstrate the full evaluation pipeline without real GT data.

print("Computing reference surface …")
ref_surface = osmnx_router.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)

ground_truth = make_synthetic_ground_truth(
    ref_surface,
    n_points=200,
    noise_std_min=3.0,  # 3-minute GPS measurement noise
    random_seed=42,
)

print(f"Ground truth: {len(ground_truth)} points")
print(ground_truth[["travel_time_min", "true_travel_time_min"]].describe().round(1))

## 2 · `score_at_points()` — single backend evaluation

In [ ]:
# Compute the friction surface to compare against the reference
s_f5 = friction_5kph.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)

scores_osmnx = score_at_points(ref_surface, ground_truth)
scores_friction = score_at_points(s_f5, ground_truth)

print("OsmnxSurfaceRouter vs GT:")
for k, v in scores_osmnx.items():
    print(f"  {k:<28} {v:.3f}" if isinstance(v, float) else f"  {k:<28} {v}")

print("\nFrictionSurfaceRouter (5 kph) vs GT:")
for k, v in scores_friction.items():
    print(f"  {k:<28} {v:.3f}" if isinstance(v, float) else f"  {k:<28} {v}")

## 3 · Pixel-wise `compare_surfaces()`

In [ ]:
pixel_comp = compare_surfaces(ref_surface, s_f5)

print("Pixel-wise comparison (osmnx vs friction_5kph):")
for k, v in pixel_comp.items():
    print(f"  {k:<25} {v:.3f}" if isinstance(v, float) else f"  {k:<25} {v}")

## 4 · `run_benchmark()` — full pipeline

In [ ]:
s_f4 = friction_4kph.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)
ens = EnsembleSurfaceRouter([osmnx_router, friction_5kph], strategy="mean")
s_ens = ens.compute_surface(DESTINATION, BBOX, RESOLUTION, MODE)

result = run_benchmark(
    routers={
        "osmnx": osmnx_router,
        "friction_5kph": friction_5kph,
        "friction_4kph": friction_4kph,
        "ensemble_mean": ens,
    },
    destination=DESTINATION,
    bbox=BBOX,
    resolution_deg=RESOLUTION,
    mode=MODE,
    ground_truth=ground_truth,
    test_fraction=0.3,
    random_seed=42,
)

print(result.summary())
print()
print("Timing (seconds per backend):")
for name, t in result.timing.items():
    print(f"  {name:<20} {t:.1f}s")

## 5 · Scores table

In [ ]:
pd.set_option("display.float_format", "{:.2f}".format)
result.scores[
    ["mae", "rmse", "bias", "r2", "coverage_error_30min", "coverage_error_60min"]
]

## 6 · In-sample vs. out-of-sample

In [ ]:
train_scores = result.scores.xs("train", level="split")
test_scores = result.scores.xs("test", level="split")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (scores, label) in zip(
    axes,
    [(train_scores, "In-sample (train)"), (test_scores, "Out-of-sample (test)")],
    strict=False,
):
    scores["mae"].plot(kind="bar", ax=ax, color="#1f77b4", alpha=0.8)
    ax.set_title(f"MAE — {label}")
    ax.set_ylabel("MAE (minutes)")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

# Overfitting check: is test MAE much worse than train MAE?
for name in train_scores.index:
    delta = test_scores.loc[name, "mae"] - train_scores.loc[name, "mae"]
    print(f"{name:<20}  ΔMAE (test−train) = {delta:+.2f} min")

## 7 · Residual maps — where does each backend fail?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
extent = [BBOX.west, BBOX.east, BBOX.south, BBOX.north]

for ax, (surface, name) in zip(
    axes,
    [
        (ref_surface, "osmnx"),
        (s_f5, "friction_5kph"),
    ],
    strict=False,
):
    resid_gdf = residual_geodataframe(surface, ground_truth)
    sc = ax.scatter(
        resid_gdf.geometry.x,
        resid_gdf.geometry.y,
        c=resid_gdf["residual_min"],
        cmap="RdBu",
        vmin=-20,
        vmax=20,
        s=40,
        alpha=0.8,
    )
    ax.scatter(
        [DESTINATION[1]], [DESTINATION[0]], c="black", s=200, marker="*", zorder=5
    )
    ax.set_xlim(BBOX.west, BBOX.east)
    ax.set_ylim(BBOX.south, BBOX.north)
    ax.set_title(f"{name}\nResiduals (pred − obs)")
    plt.colorbar(sc, ax=ax, label="Residual (min)")

plt.tight_layout()
plt.show()
print("Blue = under-predicted (faster than reality)")
print("Red  = over-predicted  (slower than reality)")

## 8 · Coverage error — the metric that matters

In [ ]:
# Coverage error: fraction of survey points where the model classifies
# access incorrectly (e.g. says "within 30 min" but observed = 45 min)

thresholds = [15, 30, 45, 60]
backends = {
    "osmnx": ref_surface,
    "friction_5kph": s_f5,
    "ensemble": s_ens,
}

rows = []
for threshold in thresholds:
    for name, surf in backends.items():
        lons = np.array([g.x for g in ground_truth.geometry])
        lats = np.array([g.y for g in ground_truth.geometry])
        from idd_travel_time.access.metrics import _sample_surface_at_points

        preds = _sample_surface_at_points(surf, ground_truth)
        obs = ground_truth["travel_time_min"].values
        valid = ~np.isnan(preds)
        pred_within = preds[valid] <= threshold
        obs_within = obs[valid] <= threshold
        cov_err = float(np.mean(pred_within != obs_within))
        rows.append(
            {"backend": name, "threshold_min": threshold, "coverage_error": cov_err}
        )

cov_df = pd.DataFrame(rows).pivot(
    index="backend", columns="threshold_min", values="coverage_error"
)
cov_df.columns = [f"{c} min" for c in cov_df.columns]
print("Coverage error (lower = better):")
print(cov_df.round(3))

## 9 · Pairwise backend comparison table

In [ ]:
print("Pairwise pixel comparisons:")
pairs_df = pd.DataFrame(result.pairwise).T[
    ["mean_diff", "rmsd", "pearson_r", "agreement_30min", "agreement_60min"]
]
pairs_df.round(3)